In [ ]:
import pandas as pd
import os
from datetime import datetime

class PowerPlants(object):
    def __init__(self):
        self.database_file = 'database.csv'

    def analyse_plant_data(self, file_path: str):
        '''
        Load the csv files: 'gas_fr_plants.csv', 'gas_plants.csv', 'wind_plants.csv'
        Clean it by: 
        1. Removing duplicate rows 
        2. Impute missing data with 0 as outlined in tasks pdf
        3. Delete -ive volumes
        4. Ensure dates are datetime objects and Volumes are numeric
        '''

        clean_df = pd.read_csv(file_path, index_col=False)

        # Remove duplicate rows
        clean_df = clean_df.drop_duplicates()

        # Convert date from str -> datetime objects
        clean_df['Date'] = pd.to_datetime(clean_df['Date'], errors='coerce', dayfirst=True)
        
        # Replace missing values with 0
        clean_df = clean_df.fillna(0)

        # Remove entire row with -ive volumes and assume NaN values are 0
        clean_df['Volume'] = pd.to_numeric(clean_df['Volume'], errors='coerce')
        clean_df['Volume'] = clean_df['Volume'].fillna(0)
        clean_df = clean_df[clean_df['Volume'] >= 0]
    
        # Remove spaces in column heading
        clean_df.columns = clean_df.columns.str.strip()

        # Reset index after removing rows
        clean_df = clean_df.reset_index(drop=True)

        return clean_df   

    def load_new_data_from_file(self, clean_df: pd.DataFrame):
        '''
        Process the cleaned dataframe (clean_df) ready for saving to 'database.csv' by:
        1. Rename Country codes to full Country name
        2. Add 'updateby' (fixed as petroineos) and 'updatetime' (current timestamp at time of updating) columns
        3. Reorder column headings
        '''

        processed_df = clean_df.copy()

        # Rename country abbreviations with full country name
        country_codes = {'GB': 'Great Britain', 'FR': 'France'}
        processed_df['Country'] = processed_df['Country'].replace(country_codes)

        # Standardise Date format before saving
        processed_df['Date'] = pd.to_datetime(processed_df['Date']).dt.strftime('%Y-%m-%d')

        # Add updateby and updatetime columns
        processed_df['updateby'] = 'petroineos'
        processed_df['updatetime'] = datetime.now()

        # Reorder column headings
        processed_df = processed_df[['Date', 'Country', 'SiteName', 'Technology', 'updateby', 'updatetime', 'Volume']]
        
        return processed_df

    def save_new_data(self, input_data: pd.DataFrame):
        ''' 
        Saves processed dataframe (processed_df) to 'database.csv'
        '''

        try:
            existing_database = pd.read_csv(self.database_file)
            updated_database = pd.concat([existing_database, input_data], ignore_index=True)
        except FileNotFoundError:
            updated_database = input_data

        updated_database.to_csv(self.database_file, index=False)
        
    def get_data_from_database(self):
        ''' 
        Read 'database.csv' and return the most recent updated record for
        every SiteName and Date combination, based on the 'updatetime' column

        This handles cases where the same plant and date was loaded more than
        once
        '''

        database_df = pd.read_csv(self.database_file)

        # Converts updatetime from str -> datetime object
        database_df['updatetime'] = pd.to_datetime(database_df['updatetime'], errors='coerce')
        database_df = database_df.sort_values('updatetime')
        
        # Keep the latest entry of data 
        latest_database_df = database_df.drop_duplicates(subset= ['Date', 'SiteName'], keep='last')

        # Lastly sort by index to keep the correct date order
        latest_database_df = (latest_database_df.sort_index())

        return latest_database_df

    def aggregate_data_to_quartely(self):
        ''' 
        Returns quarterly Volume summary statistics (mean, median, std) for each site

        Quarter mapping:
          Q1 : Jan-Mar
          Q2 : Apr-Jun
          Q3 : Jul-Sep
          Q4 : Oct-Dec
        '''

        df = self.get_data_from_database()

        # Revert back to datetime object from string format in load_new_data_from_file function
        df['Date'] = pd.to_datetime(df['Date'])

        # Aggregate Dates into Quarters (YrQ1, YrQ2, YrQ3, YrQ4)
        df['Quarter'] = df['Date'].dt.to_period('Q')

        # Aggregate database to compute summary statistics of Volume for each site per Quarter
        # with quarters as rows and SiteName as columns
        quarterly_df = pd.pivot_table(df, values='Volume', index='Quarter', columns='SiteName', aggfunc=['mean', 'median', 'std'])

        # Swap column levels so SiteName comes first, then statistic
        quarterly_df = quarterly_df.swaplevel(0, 1, axis=1)

        # Sort columns so each site has mean, median, std together
        quarterly_df = quarterly_df.sort_index(axis=1, level=0)

        return quarterly_df

    def aggregate_data_to_country(self):
        ''' 
        Returns total power production for each country by technology type
        '''

        df = self.get_data_from_database()

        # Aggregate database to compute Total Volume for each Country and Technology
        country_df = pd.pivot_table(df, values='Volume', index=['Country', 'Technology'], aggfunc='sum')

        return country_df
    


In [377]:
pp = PowerPlants()

# Delete old database before creating new one
if os.path.exists('database.csv'):
    os.remove('database.csv')

# Combines and saves each csv file into one database.csv file
csv_file = ['gas_fr_plants.csv', 'gas_plants.csv', 'wind_plants.csv']
for file in csv_file:
    new_data = pp.load_new_data_from_file(pp.analyse_plant_data(file))
    pp.save_new_data(new_data)


In [378]:
# Call get_data_from_database function
df_database = pp.get_data_from_database()
display(df_database)

,Date,Country,SiteName,Technology,updateby,updatetime,Volume
0,2024-01-01,France,Blenod-5,Gas,petroineos,2026-06-04 02:26:06.539206,6753.000000
1,2024-01-02,France,Blenod-5,Gas,petroineos,2026-06-04 02:26:06.539206,3896.000000
2,2024-01-03,France,Blenod-5,Gas,petroineos,2026-06-04 02:26:06.539206,3636.000000
3,2024-01-04,France,Blenod-5,Gas,petroineos,2026-06-04 02:26:06.539206,5138.000000
4,2024-01-05,France,Blenod-5,Gas,petroineos,2026-06-04 02:26:06.539206,5265.000000
...,...,...,...,...,...,...,...
2391,2025-04-21,Great Britain,Hornsea-2,Wind,petroineos,2026-06-04 02:26:06.624861,711.231619
2392,2025-04-22,Great Britain,Hornsea-2,Wind,petroineos,2026-06-04 02:26:06.624861,808.534585
2393,2025-04-23,Great Britain,Hornsea-2,Wind,petroineos,2026-06-04 02:26:06.624861,142.450340
2394,2025-04-24,Great Britain,Hornsea-2,Wind,petroineos,2026-06-04 02:26:06.624861,392.082184


In [379]:
# Call aggregate_data_to_quartely function
quartely_df = pp.aggregate_data_to_quartely()
display(quartely_df)

SiteName     Blenod-5                         Hornsea-1              \
                 mean  median          std         mean      median   
Quarter                                                               
2024Q1    4948.417582  4912.0  1054.780273   489.115071  491.417052   
2024Q2    4976.780220  4987.0  1130.818738   458.989280  466.683337   
2024Q3    5031.195652  4941.5  1141.124279  2199.987554  540.043540   
2024Q4    4738.282609  5048.5  1711.083400   478.276781  500.668525   
2025Q1    5053.255556  5238.5  1215.552693   522.530218  533.280148   
2025Q2    4912.200000  4461.0  1287.277813   452.714035  406.341511   

SiteName                 Hornsea-2                           Pembroke-1  \
                   std        mean      median         std         mean   
Quarter                                                                   
2024Q1      280.556287  492.801595  458.764209  285.353442  7118.450549   
2024Q2      266.749336  477.136650  425.819859  282.352554  7055.450549   
2024Q3    13021.876011  482.641581  454.421450  265.213415  7210.804348   
2024Q4      293.671778  506.687955  488.632975  285.389968  7071.673913   
2025Q1      270.010019  530.510781  571.584016  290.506335  6816.377778   
2025Q2      278.009828  551.854866  533.487056  316.884377  7011.880000   

SiteName                        Pembroke-2                       
          median          std         mean  median          std  
Quarter                                                          
2024Q1    7192.0  1081.092033  6893.472527  6824.0  1188.405826  
2024Q2    6828.0  1193.288652  7086.824176  7097.0  1156.965404  
2024Q3    7311.5  1134.721075  7041.130435  7068.0  1146.210644  
2024Q4    7294.5  1154.860232  6863.586957  6766.5  1236.674486  
2025Q1    6769.0  1100.306184  6920.877778  7361.0  1828.154044  
2025Q2    7006.0  1008.187124  6965.360000  7071.0  1300.411886

In [380]:
# Call aggregate_data_to_country function
country_sum = pp.aggregate_data_to_country()
display(country_sum)

Volume
Country       Technology              
France        Gas         2.379583e+06
Great Britain Gas         6.741038e+06
              Wind        6.274046e+05